In [1]:
import requests
import json
import time
import sdmx
import pandas as pd
import numpy as np
import yfinance as yf
from pyjstat import pyjstat
from ecbdata import ecbdata
from itertools import islice
from io import StringIO

In [37]:
#Country Map for Data Modelling

country_map = {
    "Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)": "EA",
    "European Union (EU6-1958, EU9-1973, EU10-1981, EU12-1986, EU15-1995, EU25-2004, EU27-2007, EU28-2013, EU27-2020)": "EU",
    "European Union - 28 countries (2013-2020)": "EU28",
    "European Union - 27 countries (from 2020)": "EU27-2020",
    "European Union - 27 countries (2007-2013)": "EU27-07-13",
    "Euro area – 21 countries (from 2026)": "EA21",
    "Euro area – 20 countries (2023-2025)": "EA20",
    "Euro area - 19 countries  (2015-2022)": "EA19",
    "Euro area - 18 countries (2014)": "EA18",
    "Euro area - 17 countries (2011-2013)": "EA17",
    "Euro area - 12 countries (2001-2006)": "EA12",
    "Belgium": "BE",
    "Germany": "DE",
    "Germany including former GDR": "DE",
    "France": "FR",
    "Spain": "ES",
    "Italy": "IT",
    "Liechtenstein": "LI",
    "Netherlands": "NL",
    "Greece": "GR",
    "Austria": "AT",
    "Poland": "PL",
    "Portugal": "PT",
    "Sweden": "SE",
    "Finland": "FI",
    "Ireland": "IE",
    "Czechia": "CZ",
    "Denmark": "DK",
    "Estonia": "EE",
    "Latvia": "LV",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Hungary": "HU",
    "Slovakia": "SK",
    "Slovenia": "SI",
    "Bulgaria": "BG",
    "Romania": "RO",
    "Croatia": "HR",
    "United Kingdom": "UK",
    "Türkiye": "TR",
    "United States": "US",
    "Japan": "JP",
    "Cyprus": "CY",
    "Malta": "MT",
    "Iceland": "IS",
    "Norway": "NO",
    "Switzerland": "CH",
    "Montenegro": "ME",
    "North Macedonia": "MK",
    "Albania": "AL",
    "Serbia": "RS",
    "Bosnia and Herzegovina": "BA",
    "Ukraine": "UA",
    "Kosovo*": "XK",
    "Moldova": "MD",
    "Georgia": "GE"
    # weitere gemäß Eurostat-Liste ergänzen
}


euro_countries = [
    "BE", "DE", "FR", "ES", "IT", "NL", "AT", "PT", "FI", "IE",
    "GR", "LU", "CY", "MT", "SI", "SK", "EE", "LV", "LT", "HR", "EU27-2020"
]

reverse_country_map = {
    "EA": "Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)",
    "EU": "European Union (EU6-1958, EU9-1973, EU10-1981, EU12-1986, EU15-1995, EU25-2004, EU27-2007, EU28-2013, EU27-2020)",
    "EU28": "European Union - 28 countries (2013-2020)",
    "EU27-2020": "European Union - 27 countries (from 2020)",
    "EU27-07-13": "European Union - 27 countries (2007-2013)",
    "EA21": "Euro area – 21 countries (from 2026)",
    "EA20": "Euro area – 20 countries (2023-2025)",
    "EA19": "Euro area - 19 countries  (2015-2022)",
    "EA18": "Euro area - 18 countries (2014)",
    "EA17": "Euro area - 17 countries (2011-2013)",
    "EA12": "Euro area - 12 countries (2001-2006)",
    "BE": "Belgium",
    "DE": "Germany",
    "FR": "France",
    "ES": "Spain",
    "IT": "Italy",
    "LI": "Liechtenstein",
    "NL": "Netherlands",
    "GR": "Greece",
    "AT": "Austria",
    "PL": "Poland",
    "PT": "Portugal",
    "SE": "Sweden",
    "FI": "Finland",
    "IE": "Ireland",
    "CZ": "Czechia",
    "DK": "Denmark",
    "EE": "Estonia",
    "LV": "Latvia",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "HU": "Hungary",
    "SK": "Slovakia",
    "SI": "Slovenia",
    "BG": "Bulgaria",
    "RO": "Romania",
    "HR": "Croatia",
    "UK": "United Kingdom",
    "TR": "Türkiye",
    "US": "United States",
    "JP": "Japan",
    "CY": "Cyprus",
    "MT": "Malta",
    "IS": "Iceland",
    "NO": "Norway",
    "CH": "Switzerland",
    "ME": "Montenegro",
    "MK": "North Macedonia",
    "AL": "Albania",
    "RS": "Serbia",
    "BA": "Bosnia and Herzegovina",
    "UA": "Ukraine",
    "XK": "Kosovo*",
    "MD": "Moldova",
    "GE": "Georgia"
}

euro_intro_year = {
    "BE": 1999, "DE": 1999, "IE": 1999, "GR": 2001, "ES": 1999, "FR": 1999,
    "IT": 1999, "CY": 2008, "LU": 1999, "MT": 2008, "NL": 1999, "AT": 1999,
    "PT": 1999, "SI": 2007, "SK": 2009, "EE": 2011, "LV": 2014, "LT": 2015,
    "HR": 2023, "EU27-2020": 1990
}


ea20_countries = [
    "AT", "BE", "CY", "DE", "EE", "ES", "FI", "FR", "GR", "HR",
    "IE", "IT", "LT", "LU", "LV", "MT", "NL", "PT", "SI", "SK"
]


## Downloading Money Market Interest Rate Data (Yearly, Total)

In [3]:
# Defining Eurostats URL, here: yearly Data
#url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/irt_st_a?format=JSON&int_rt=IRT_DTD&int_rt=IRT_M1&int_rt=IRT_M12&int_rt=IRT_M3&int_rt=IRT_M6&lang=EN"
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/irt_st_a?format=JSON&lang=EN&int_rt=IRT_M3"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
mm_ir_yy_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
mm_ir_yy_df['value'] = pd.to_numeric(mm_ir_yy_df['value'], errors='coerce')

# Data Cleaning and Transformation
mm_ir_yy_df = mm_ir_yy_df.dropna(subset=["value"])
mm_ir_yy_df = mm_ir_yy_df.rename(columns={"value": "Interest Rate", "Time": "Year"})
mm_ir_yy_df = mm_ir_yy_df.drop(columns=["Time frequency", "Interest rate"])
mm_ir_yy_df["Country Code"] = mm_ir_yy_df["Geopolitical entity (reporting)"].map(country_map)
mm_ir_yy_df["YearCountryKey"] = mm_ir_yy_df["Year"].astype(str) + mm_ir_yy_df["Country Code"]
mm_ir_yy_df['Year'] = pd.to_numeric(mm_ir_yy_df['Year'], errors='coerce')

# -- ----------------------------------------

# Euro Area Werte extrahieren
ea_rates = (
    mm_ir_yy_df[mm_ir_yy_df["Country Code"] == "EA"]
    .set_index("Year")["Interest Rate"]
)

# Expanding the Euro Area rows to include all individual Euro Area countries
rows = []
for cc, start_year in euro_intro_year.items():
    for year in range(start_year, 2026):
        rows.append([cc, year])

expanded_df = pd.DataFrame(rows, columns=["Country Code", "Year"])

# Adding the expanded Euro Area rows back to the main DataFrame
expanded_df = expanded_df.merge(
    ea_rates.rename("Interest Rate"),
    left_on="Year",
    right_index=True,
    how="left"
)

# Adding Geopolitical entity (reporting) and YearCountryKey columns
expanded_df["Geopolitical entity (reporting)"] = expanded_df["Country Code"].map(reverse_country_map)
expanded_df["YearCountryKey"] = expanded_df["Year"].astype(str) + expanded_df["Country Code"]

# -- ----------------------------------------

# Identifying existing YearCountryKeys
existing_keys = set(mm_ir_yy_df["YearCountryKey"])

# Filtering out rows from expanded_df that already exist in mm_ir_yy_df
expanded_new = expanded_df[~expanded_df["YearCountryKey"].isin(existing_keys)]

# Adding only the new rows to mm_ir_yy_df
mm_ir_yy_df = pd.concat([mm_ir_yy_df, expanded_new], ignore_index=True)

print(mm_ir_yy_df.head())

                     Geopolitical entity (reporting)  Year  Interest Rate  \
0  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  1990       10.37460   
1  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  1991       10.32792   
2  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  1992       10.91888   
3  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  1993        8.52342   
4  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  1994        6.52500   

  Country Code YearCountryKey  
0           EA         1990EA  
1           EA         1991EA  
2           EA         1992EA  
3           EA         1993EA  
4           EA         1994EA  


In [5]:
mm_ir_yy_df.to_csv("Money_Market_Interest_Rate.csv", index=False, sep=';', decimal=',')

## Downloading real GDP (2015 = 100) per capita data (Yearly, Total)

In [4]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tipsna40?format=JSON&unit=CLV15_EUR_HAB&na_item=B1GQ&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
gdp_pc_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
gdp_pc_df['value'] = pd.to_numeric(gdp_pc_df['value'], errors='coerce')

# Data Cleaning and Transformation
gdp_pc_df = gdp_pc_df.dropna(subset=["value"])
gdp_pc_df = gdp_pc_df.rename(columns={"value": "Real GDP (2015 = 100) (in EUR) per Capita", "Time": "Year"})
gdp_pc_df = gdp_pc_df.drop(columns=["Time frequency", "Unit of measure"])
gdp_pc_df["Country Code"] = gdp_pc_df["Geopolitical entity (reporting)"].map(country_map)
gdp_pc_df["YearCountryKey"] = gdp_pc_df["Year"].astype(str) + gdp_pc_df["Country Code"]

print(gdp_pc_df.head())

    National accounts indicator (ESA 2010)  \
0  Gross domestic product at market prices   
1  Gross domestic product at market prices   
2  Gross domestic product at market prices   
3  Gross domestic product at market prices   
4  Gross domestic product at market prices   

             Geopolitical entity (reporting)  Year  \
0  European Union - 27 countries (from 2020)  1995   
1  European Union - 27 countries (from 2020)  1996   
2  European Union - 27 countries (from 2020)  1997   
3  European Union - 27 countries (from 2020)  1998   
4  European Union - 27 countries (from 2020)  1999   

   Real GDP (2015 = 100) (in EUR) per Capita Country Code YearCountryKey  
0                                    20910.0    EU27-2020  1995EU27-2020  
1                                    21260.0    EU27-2020  1996EU27-2020  
2                                    21810.0    EU27-2020  1997EU27-2020  
3                                    22430.0    EU27-2020  1998EU27-2020  
4                      

In [7]:
gdp_pc_df.to_csv("GDP (in EUR) per Capita.csv", index=False, sep=';', decimal=',')

# EU 27 is European Union - 27 countries (from 2020)
# EA 21 is Euro area - 21 countries (from 2026)
# EA20 is Euro area – 20 countries (2023-2025)

## Download Imports of Goods and Services in %of GDP (Yearly, Categorical, total included to economic df)

In [5]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tet00004?format=JSON&unit=PC_GDP&na_item=P71&na_item=P72&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
imports_percgdp_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
imports_percgdp_df['value'] = pd.to_numeric(imports_percgdp_df['value'], errors='coerce')

# Data Cleaning and Transformation
imports_percgdp_df = imports_percgdp_df.dropna(subset=["value"])
imports_percgdp_df = imports_percgdp_df.rename(columns={"value": "Imports as Percentage of GDP", "Time": "Year"})
imports_percgdp_df = imports_percgdp_df.drop(columns=["Time frequency", "Unit of measure"])
imports_percgdp_df["Country Code"] = imports_percgdp_df["Geopolitical entity (reporting)"].map(country_map)
imports_percgdp_df["YearCountryKey"] = imports_percgdp_df["Year"].astype(str) + imports_percgdp_df["Country Code"]

print(imports_percgdp_df.head())

  National accounts indicator (ESA 2010)  \
0                       Imports of goods   
1                       Imports of goods   
2                       Imports of goods   
3                       Imports of goods   
4                       Imports of goods   

             Geopolitical entity (reporting)  Year  \
0  European Union - 27 countries (from 2020)  2014   
1  European Union - 27 countries (from 2020)  2015   
2  European Union - 27 countries (from 2020)  2016   
3  European Union - 27 countries (from 2020)  2017   
4  European Union - 27 countries (from 2020)  2018   

   Imports as Percentage of GDP Country Code YearCountryKey  
0                          30.4    EU27-2020  2014EU27-2020  
1                          30.0    EU27-2020  2015EU27-2020  
2                          29.5    EU27-2020  2016EU27-2020  
3                          30.9    EU27-2020  2017EU27-2020  
4                          32.0    EU27-2020  2018EU27-2020  


In [ ]:
imports_percgdp_df.to_csv("Imports as Percentage of GDP.csv", index=False, sep=';', decimal=',')

## Download Foreign Value Added (Yearly, Categorical, Value devided by importing nation, total included to economic df)
### Value added to products that were imported from non EU countries and exported to non EU countries

In [6]:
# -----------------------------
# Parameter
# -----------------------------
BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/naio_10_fgfoee"
# Fixe Filter (je nach Bedarf anpassen)
FIXED_PARAMS = {
    "format": "JSON",
    "c_imp": "NEU27_2020",   # Importeur (EU27_2020)
    "lang": "EN"
}

# Liste der NACE-Kategorien (deiner URL entnommen)
NACE_VALUES = [
    "A","A01","A02","A03","B","B-E","B-F","C","C10-C12","C13-C15","C16","C17","C18",
    "C19","C20","C21","C22","C23","C24","C25","C26","C27","C28","C29","C30","C31_C32",
    "C33","D35","E36","E37-E39","F","G-I","G-U","G45","G46","G47","H49","H50","H51",
    "H52","H53","I","J","J58","J59_J60","J61","J62_J63","K","K64","K65","K66","L",
    "M_N","M69_M70","M71","M72","M73","M74_M75","N77","N78","N79","N80-N82","O-Q",
    "O84","P85","Q86","Q87_Q88","R-U","R90-R92","R93","S94","S95","S96","T","TOTAL","U"
]
# nace_r2 = Wirtschaftszweig (Industrie-/Sektor-Code) nach Eurostats Klassifikation

# Liste der Herkunftsländer (der URL von Eurostats Query Builder entnommen)
C_ORIG_VALUES = [
    "AL","AR","AU","BR","CA","CH","CN","ID","IN","JP","KR","ME","MK","MX",
    "NO","RS","RU","SA","TR","UK","US","WRL_REST","ZA"
]
# "NEU27_2020",

# Batchgrößen (je nach Stabilität/URL-Länge anpassen)
NACE_BATCH_SIZE = 12
ORIG_BATCH_SIZE = 10

# Timeout & Retry-Einstellungen
REQUEST_TIMEOUT = 60  # Sekunden
MAX_RETRIES = 3
RETRY_SLEEP = 2  # Sekunden zwischen Retries


# -----------------------------
# Hilfsfunktionen
# -----------------------------
def batched(iterable, n):               # Teile eine Liste/Iterable in Batches der Größe n.
    it = iter(iterable)                 # Ein Iterator erlaubt es, die Elemente einzeln, nacheinander abzurufen
    while True:
        batch = list(islice(it, n))     # Gibt Listen der Länge n zurück, bis der Iterator erschöpft ist
        if not batch:
            return
        yield batch                     # Yield batch gibt die einzelnen Listen als Resultat her, bis die while Schleife
                                        # "not batch" erreicht und die Funktion mit return verlassen wird

def build_params(nace_list, c_orig_list):
    """
    KI: Erzeuge die Query-Parameter für eine Anfrage.
    Mehrfachwerte werden mehrfach als Schlüssel gesetzt (requests unterstützt das über tuples).
    """
    params = []
    # fixe Parameter
    for k, v in FIXED_PARAMS.items():
        params.append((k, v))
    # mehrere nace_r2
    for v in nace_list:
        params.append(("nace_r2", v))
    # mehrere c_orig
    for v in c_orig_list:
        params.append(("c_orig", v))
    return params

def fetch_batch(nace_batch, orig_batch, session=None):
    """Hole einen Batch (nace_r2 × c_orig) mit Retry-Logik und parse via pyjstat."""
    sess = session or requests.Session()                # Wenn session NULL ist, ist sess als request.Session() definiert
    params = build_params(nace_batch, orig_batch)
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = sess.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            # JSON-stat in DataFrame
            json_string = json.dumps(resp.json())
            dataset = pyjstat.Dataset.read(json_string)
            df = dataset.write('dataframe')
            # Optional: Spalte für Nachvollziehbarkeit, welcher Batch geladen wurde
            # df["batch_nace"] = ",".join(nace_batch)
            # df["batch_c_orig"] = ",".join(orig_batch)
            return df
        except Exception as e:              # Wenn try nicht funktioniert, startet except. e ist hier eine Variable mit dem Fehlercode
            last_exc = e                    
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP)     # Tritt ein Fehler auf, wartet die Funktion RETRY_SLEEP Sekunden und versucht es erneut, bis MAX_RETRIES erreicht ist
            else:
                raise last_exc              # Wenn der Fehler nach Max Retries weiter besteht, wird die Funktion abgebrochen und der Fehlercode wird ausgegeben

# -----------------------------
# Hauptlogik: alle Batches laden und zusammenführen
# -----------------------------
all_parts = []
with requests.Session() as session:
    for nace_batch in batched(NACE_VALUES, NACE_BATCH_SIZE):
        for orig_batch in batched(C_ORIG_VALUES, ORIG_BATCH_SIZE):
            df_part = fetch_batch(nace_batch, orig_batch, session=session)
            # Leere Ergebnisse können vorkommen; nur sinnvolle Teile sammeln
            if df_part is not None and not df_part.empty:
                all_parts.append(df_part)

# Zusammenführen
if not all_parts:
    raise RuntimeError("Keine Daten erhalten – prüfe Filter/Parameter.")
fva_df = pd.concat(all_parts, ignore_index=True)

# Defining values to be numeric to avoid issues later on
fva_df['value'] = pd.to_numeric(fva_df['value'], errors='coerce')

# Optional: Duplikate entfernen (je nach Struktur; typisch sind Spalten wie 'time', 'geo', 'nace_r2', 'c_orig', 'value', etc.)
# Setze hier einen sinnvollen Satz an Schlüsseln – Beispiel:
possible_keys = [c for c in ["time", "time_period", "geo", "nace_r2", "c_orig", "c_imp", "indic_na", "unit", "partner"] if c in fva_df.columns]
if possible_keys:
    fva_df = fva_df.drop_duplicates(subset=possible_keys + ([ "value" ] if "value" in fva_df.columns else []))

# Data Cleaning and Transformation
fva_df = fva_df.dropna(subset=["value"])
fva_df = fva_df.rename(columns={"value": "Foreign Value Added (Thousand EUR)", "Time": "Year"})
fva_df = fva_df.drop(columns=["Time frequency", "Unit of measure", "Country of import"])
fva_df["Country Code"] = fva_df["Geopolitical entity (reporting)"].map(country_map)
fva_df["YearCountryKey"] = fva_df["Year"].astype(str) + fva_df["Country Code"]

print(fva_df.head())
print(f"Form: {fva_df.shape}")

  Statistical classification of economic activities in the European Community (NACE Rev. 2)  \
0                  Agriculture, forestry and fishing                                          
1                  Agriculture, forestry and fishing                                          
2                  Agriculture, forestry and fishing                                          
3                  Agriculture, forestry and fishing                                          
4                  Agriculture, forestry and fishing                                          

  Country of origin            Geopolitical entity (reporting)  Year  \
0       Switzerland  European Union - 27 countries (from 2020)  2010   
1       Switzerland  European Union - 27 countries (from 2020)  2011   
2       Switzerland  European Union - 27 countries (from 2020)  2012   
3       Switzerland  European Union - 27 countries (from 2020)  2013   
4       Switzerland  European Union - 27 countries (from 2020)  2014 

In [11]:
fva_df.to_csv("Foreign_Value_Added.csv", index=False, sep=';', decimal=',')

## Download Inflation

### HICP EUR Area (Yearly, Total)
#### Harmonized Index Consumer Prices (annual average rate of change)

In [96]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tec00118?format=JSON&unit=RCH_A_AVG&coicop18=TOTAL&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
hicp_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
hicp_df['value'] = pd.to_numeric(hicp_df['value'], errors='coerce')

# Data Cleaning and Transformation
hicp_df = hicp_df.dropna(subset=["value"])
hicp_df = hicp_df.rename(columns={"value": "Harmonized Index of Consumer Prices (HICP, annual average rate of change)", "Time": "Year"})
hicp_df = hicp_df.drop(columns=["Time frequency", "Unit of measure", "Classification of individual consumption by purpose (COICOP) - 2018"])
hicp_df["Country Code"] = hicp_df["Geopolitical entity (reporting)"].map(country_map)
hicp_df["YearCountryKey"] = hicp_df["Year"].astype(str) + hicp_df["Country Code"]

hicp_df["Year"] = pd.to_numeric(hicp_df["Year"], errors="coerce")

# --- Konfiguration ---
value_col = "Harmonized Index of Consumer Prices (HICP, annual average rate of change)"

ea_label = (
    "Euro area (EA11-1999, EA12-2001, EA13-2007, EA15-2008, "
    "EA16-2009, EA17-2011, EA18-2014, EA19-2015, EA20-2023, EA21-2026)"
)

# --- 1. Vorhandene EA-Zeilen entfernen ---
hicp_df = hicp_df[hicp_df["Country Code"] != "EA"].copy()

# --- 2. Hilfsfunktion zum Erzeugen der EA-Zeilen ---
def build_ea_block(df, source_code, year_min=None, year_max=None):
    block = df[df["Country Code"] == source_code].copy()

    if year_min is not None:
        block = block[block["Year"] >= year_min]
    if year_max is not None:
        block = block[block["Year"] <= year_max]

    block["Country Code"] = "EA"
    block["Geopolitical entity (reporting)"] = ea_label
    block["YearCountryKey"] = block["Year"].astype(str) + "EA"

    return block


# --- 3. EA nach Zeitabschnitten aufbauen ---
ea_2001_2014 = build_ea_block(
    hicp_df,
    source_code="EA12",
    year_min=2001,
    year_max=2014
)

ea_2015_2022 = build_ea_block(
    hicp_df,
    source_code="EA19",
    year_min=2015,
    year_max=2022
)

ea_2023_plus = build_ea_block(
    hicp_df,
    source_code="EA20",
    year_min=2023
)

# --- 4. Alle EA-Zeilen zusammenführen ---
ea_df = pd.concat(
    [ea_2001_2014, ea_2015_2022, ea_2023_plus],
    ignore_index=True
)

# --- 5. Zur Ausgangstabelle hinzufügen ---
hicp_df = pd.concat([hicp_df, ea_df], ignore_index=True)

print(hicp_df.head())

             Geopolitical entity (reporting)  Year  \
0  European Union - 27 countries (from 2020)  2014   
1  European Union - 27 countries (from 2020)  2015   
2  European Union - 27 countries (from 2020)  2016   
3  European Union - 27 countries (from 2020)  2017   
4  European Union - 27 countries (from 2020)  2018   

   Harmonized Index of Consumer Prices (HICP, annual average rate of change)  \
0                                                0.4                           
1                                                0.1                           
2                                                0.2                           
3                                                1.6                           
4                                                1.8                           

  Country Code YearCountryKey  
0    EU27-2020  2014EU27-2020  
1    EU27-2020  2015EU27-2020  
2    EU27-2020  2016EU27-2020  
3    EU27-2020  2017EU27-2020  
4    EU27-2020  2018EU27-2020  


In [97]:
hicp_df.to_csv("HICP.csv", index=False, sep=';', decimal=',')

### HICP contribution to Inflation Rate (in %) EUR Area Monthly Data (Yearly, Categorical)

In [8]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/prc_hicp_ctr?format=JSON&unit=PC_PNT&coicop18=CP01&coicop18=CP011&coicop18=CP0111&coicop18=CP01111&coicop18=CP01112&coicop18=CP01113&coicop18=CP01114&coicop18=CP01115&coicop18=CP01119&coicop18=CP0112&coicop18=CP01122&coicop18=CP01123&coicop18=CP01124&coicop18=CP01125&coicop18=CP0113&coicop18=CP01131&coicop18=CP01132&coicop18=CP01133&coicop18=CP01134&coicop18=CP01135&coicop18=CP01136&coicop18=CP01137&coicop18=CP0114&coicop18=CP01141&coicop18=CP01142&coicop18=CP01143&coicop18=CP01144&coicop18=CP01145&coicop18=CP01146&coicop18=CP01147&coicop18=CP01148&coicop18=CP01149&coicop18=CP0115&coicop18=CP01151&coicop18=CP01152&coicop18=CP01153&coicop18=CP01159&coicop18=CP0116&coicop18=CP01161&coicop18=CP01162&coicop18=CP01163&coicop18=CP01164&coicop18=CP01165&coicop18=CP01166&coicop18=CP01167&coicop18=CP01168&coicop18=CP01169&coicop18=CP0117&coicop18=CP01171&coicop18=CP01172&coicop18=CP01173&coicop18=CP01174&coicop18=CP01175&coicop18=CP01176&coicop18=CP01177&coicop18=CP01178&coicop18=CP01179&coicop18=CP0118&coicop18=CP01181&coicop18=CP01182&coicop18=CP01183&coicop18=CP01184&coicop18=CP01185&coicop18=CP01186&coicop18=CP01189&coicop18=CP0119&coicop18=CP01191&coicop18=CP01192&coicop18=CP01193&coicop18=CP01194&coicop18=CP01199&coicop18=CP012&coicop18=CP0121&coicop18=CP01210&coicop18=CP0122&coicop18=CP01220&coicop18=CP0123&coicop18=CP01230&coicop18=CP0124&coicop18=CP01240&coicop18=CP0125&coicop18=CP01250&coicop18=CP0126&coicop18=CP01260&coicop18=CP0129&coicop18=CP01290&coicop18=CP02&coicop18=CP021&coicop18=CP0211&coicop18=CP02110&coicop18=CP0212&coicop18=CP02121&coicop18=CP02122&coicop18=CP0213&coicop18=CP02130&coicop18=CP0219&coicop18=CP02190&coicop18=CP023&coicop18=CP0230&coicop18=CP02301&coicop18=CP02302&coicop18=CP02309&coicop18=CP03&coicop18=CP031&coicop18=CP0311&coicop18=CP03110&coicop18=CP0312&coicop18=CP03121&coicop18=CP03122&coicop18=CP03123&coicop18=CP03124&coicop18=CP0313&coicop18=CP03131&coicop18=CP03132&coicop18=CP0314&coicop18=CP03141&coicop18=CP03142&coicop18=CP032&coicop18=CP0321&coicop18=CP03211&coicop18=CP03212&coicop18=CP03213&coicop18=CP0322&coicop18=CP03220&coicop18=CP04&coicop18=CP041&coicop18=CP0411&coicop18=CP04110&coicop18=CP0412&coicop18=CP04121&coicop18=CP04122&coicop18=CP043&coicop18=CP0431&coicop18=CP04311&coicop18=CP04312&coicop18=CP0432&coicop18=CP04320&coicop18=CP044&coicop18=CP0441&coicop18=CP04411&coicop18=CP0442&coicop18=CP04420&coicop18=CP0443&coicop18=CP04431&coicop18=CP04432&coicop18=CP0444&coicop18=CP04441&coicop18=CP04449&coicop18=CP045&coicop18=CP0451&coicop18=CP04510&coicop18=CP0452&coicop18=CP04521&coicop18=CP04522&coicop18=CP0453&coicop18=CP04530&coicop18=CP0454&coicop18=CP04541&coicop18=CP04542&coicop18=CP04543&coicop18=CP04549&coicop18=CP0455&coicop18=CP04550&coicop18=CP05&coicop18=CP051&coicop18=CP0511&coicop18=CP05111&coicop18=CP05112&coicop18=CP05113&coicop18=CP05114&coicop18=CP0512&coicop18=CP05120&coicop18=CP052&coicop18=CP0521&coicop18=CP05211&coicop18=CP05212&coicop18=CP05213&coicop18=CP05219&coicop18=CP053&coicop18=CP0531&coicop18=CP05311&coicop18=CP05312&coicop18=CP05313&coicop18=CP05314&coicop18=CP05319&coicop18=CP0532&coicop18=CP05321&coicop18=CP05322&coicop18=CP05329&coicop18=CP0533&coicop18=CP05330&coicop18=CP054&coicop18=CP0540&coicop18=CP05401&coicop18=CP05402&coicop18=CP05403&coicop18=CP055&coicop18=CP0551&coicop18=CP05510&coicop18=CP0552&coicop18=CP05521&coicop18=CP05522&coicop18=CP0553&coicop18=CP05530&coicop18=CP056&coicop18=CP0561&coicop18=CP05611&coicop18=CP05619&coicop18=CP0562&coicop18=CP05621&coicop18=CP05629&coicop18=CP06&coicop18=CP061&coicop18=CP0611&coicop18=CP06111&coicop18=CP06112&coicop18=CP0612&coicop18=CP06121&coicop18=CP06122&coicop18=CP06123&coicop18=CP0613&coicop18=CP06131&coicop18=CP06132&coicop18=CP06133&coicop18=CP0614&coicop18=CP06140&coicop18=CP062&coicop18=CP0621&coicop18=CP06211&coicop18=CP06219&coicop18=CP0622&coicop18=CP06221&coicop18=CP06229&coicop18=CP0623&coicop18=CP06231&coicop18=CP06232&coicop18=CP063&coicop18=CP0631&coicop18=CP06310&coicop18=CP0632&coicop18=CP06320&coicop18=CP064&coicop18=CP0641&coicop18=CP06410&coicop18=CP0642&coicop18=CP06420&coicop18=CP07&coicop18=CP071&coicop18=CP0711&coicop18=CP07111&coicop18=CP07112&coicop18=CP0712&coicop18=CP07120&coicop18=CP0713&coicop18=CP07130&coicop18=CP072&coicop18=CP0721&coicop18=CP07211&coicop18=CP07212&coicop18=CP07213&coicop18=CP0722&coicop18=CP07221&coicop18=CP07222&coicop18=CP07223&coicop18=CP07224&coicop18=CP0723&coicop18=CP07230&coicop18=CP0724&coicop18=CP07241&coicop18=CP07242&coicop18=CP07243&coicop18=CP07244&coicop18=CP073&coicop18=CP0731&coicop18=CP07311&coicop18=CP07312&coicop18=CP0732&coicop18=CP07321&coicop18=CP07322&coicop18=CP07323&coicop18=CP0733&coicop18=CP07331&coicop18=CP07332&coicop18=CP0734&coicop18=CP07340&coicop18=CP0735&coicop18=CP07350&coicop18=CP0736&coicop18=CP07360&coicop18=CP074&coicop18=CP0741&coicop18=CP07411&coicop18=CP07412&coicop18=CP0749&coicop18=CP07491&coicop18=CP07492&coicop18=CP08&coicop18=CP081&coicop18=CP0811&coicop18=CP08110&coicop18=CP0812&coicop18=CP08120&coicop18=CP0813&coicop18=CP08131&coicop18=CP08132&coicop18=CP0814&coicop18=CP08140&coicop18=CP0815&coicop18=CP08150&coicop18=CP0819&coicop18=CP08191&coicop18=CP08192&coicop18=CP082&coicop18=CP0820&coicop18=CP08200&coicop18=CP083&coicop18=CP0831&coicop18=CP08310&coicop18=CP0832&coicop18=CP08320&coicop18=CP0833&coicop18=CP08330&coicop18=CP0834&coicop18=CP08340&coicop18=CP0835&coicop18=CP08350&coicop18=CP0839&coicop18=CP08391&coicop18=CP08392&coicop18=CP08399&coicop18=CP09&coicop18=CP091&coicop18=CP0911&coicop18=CP09111&coicop18=CP09112&coicop18=CP09113&coicop18=CP0912&coicop18=CP09121&coicop18=CP09123&coicop18=CP09124&coicop18=CP09129&coicop18=CP092&coicop18=CP0921&coicop18=CP09211&coicop18=CP09212&coicop18=CP09213&coicop18=CP0922&coicop18=CP09221&coicop18=CP09222&coicop18=CP093&coicop18=CP0931&coicop18=CP09311&coicop18=CP09312&coicop18=CP0932&coicop18=CP09321&coicop18=CP09322&coicop18=CP094&coicop18=CP0941&coicop18=CP09410&coicop18=CP0942&coicop18=CP09421&coicop18=CP09422&coicop18=CP0943&coicop18=CP09431&coicop18=CP09432&coicop18=CP0944&coicop18=CP09440&coicop18=CP0945&coicop18=CP09450&coicop18=CP0946&coicop18=CP09461&coicop18=CP09462&coicop18=CP09463&coicop18=CP095&coicop18=CP0951&coicop18=CP09510&coicop18=CP0952&coicop18=CP09520&coicop18=CP096&coicop18=CP0961&coicop18=CP09610&coicop18=CP0962&coicop18=CP09620&coicop18=CP0963&coicop18=CP09630&coicop18=CP0969&coicop18=CP09690&coicop18=CP097&coicop18=CP0971&coicop18=CP09711&coicop18=CP09719&coicop18=CP0972&coicop18=CP09721&coicop18=CP09722&coicop18=CP0973&coicop18=CP09730&coicop18=CP0974&coicop18=CP09740&coicop18=CP098&coicop18=CP0980&coicop18=CP09800&coicop18=CP10&coicop18=CP101&coicop18=CP1010&coicop18=CP10101&coicop18=CP10102&coicop18=CP102&coicop18=CP1020&coicop18=CP10200&coicop18=CP103&coicop18=CP1030&coicop18=CP10300&coicop18=CP104&coicop18=CP1040&coicop18=CP10400&coicop18=CP105&coicop18=CP1050&coicop18=CP10501&coicop18=CP10509&coicop18=CP11&coicop18=CP111&coicop18=CP1111&coicop18=CP11111&coicop18=CP11112&coicop18=CP1112&coicop18=CP11121&coicop18=CP11129&coicop18=CP112&coicop18=CP1120&coicop18=CP11201&coicop18=CP11202&coicop18=CP11203&coicop18=CP12&coicop18=CP121&coicop18=CP1211&coicop18=CP12110&coicop18=CP1212&coicop18=CP12120&coicop18=CP1213&coicop18=CP12130&coicop18=CP1214&coicop18=CP12141&coicop18=CP12142&coicop18=CP1219&coicop18=CP12190&coicop18=CP122&coicop18=CP1222&coicop18=CP12220&coicop18=CP1229&coicop18=CP12291&coicop18=CP12299&coicop18=CP13&coicop18=CP131&coicop18=CP1311&coicop18=CP13111&coicop18=CP1312&coicop18=CP13120&coicop18=CP1313&coicop18=CP13131&coicop18=CP13132&coicop18=CP132&coicop18=CP1321&coicop18=CP13211&coicop18=CP13212&coicop18=CP1322&coicop18=CP13220&coicop18=CP1329&coicop18=CP13291&coicop18=CP13292&coicop18=CP133&coicop18=CP1330&coicop18=CP13301&coicop18=CP13302&coicop18=CP13303&coicop18=CP139&coicop18=CP1390&coicop18=CP13902&coicop18=CP13909&coicop18=FOOD&coicop18=FOOD_NP&coicop18=FOOD_P&coicop18=IGD_NNRG&coicop18=NRG&coicop18=SERV&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
hicp_perc_infl_eur_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
hicp_perc_infl_eur_df['value'] = pd.to_numeric(hicp_perc_infl_eur_df['value'], errors='coerce')

# Data Cleaning and Transformation
hicp_perc_infl_eur_df = hicp_perc_infl_eur_df.dropna(subset=["value"])
hicp_perc_infl_eur_df = hicp_perc_infl_eur_df.rename(columns={"value": "HICP Contribution to Inflation in PP", "Time": "Month"})
hicp_perc_infl_eur_df = hicp_perc_infl_eur_df.drop(columns=["Time frequency", "Unit of measure"])
hicp_perc_infl_eur_df["Country Code"] = hicp_perc_infl_eur_df["Geopolitical entity (reporting)"].map(country_map)
hicp_perc_infl_eur_df["YearCountryKey"] = hicp_perc_infl_eur_df["Month"].astype(str).str[:4] + hicp_perc_infl_eur_df["Country Code"]

print(hicp_perc_infl_eur_df.head())

  Classification of individual consumption by purpose (COICOP) - 2018  \
0                   Food and non-alcoholic beverages                    
1                   Food and non-alcoholic beverages                    
2                   Food and non-alcoholic beverages                    
3                   Food and non-alcoholic beverages                    
4                   Food and non-alcoholic beverages                    

                     Geopolitical entity (reporting)    Month  \
0  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  2002-01   
1  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  2002-02   
2  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  2002-03   
3  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  2002-04   
4  Euro area (EA11-1999, EA12-2001, EA13-2007, EA...  2002-05   

   HICP Contribution to Inflation in PP Country Code YearCountryKey  
0                                  0.93           EA         2002EA  
1                             

In [15]:
hicp_perc_infl_eur_df.to_csv("HICP Contribution to Inflation EA in PP.csv", index=False, sep=';', decimal=',')

## Download Salaries

### Download Exchange Rates

In [9]:
def get_fx_data(currencies, start_year=1999):
    print("Rufe Daten von der EZB ab...")
    ecb = sdmx.Request('ECB')
    
    # 1. EZB Daten laden
    try:
        data_response = ecb.data(resource_id='EXR', key={'CURRENCY': currencies, 'FREQ': 'A'})
        df_ecb = sdmx.to_pandas(data_response.data[0]).reset_index()
        
        # Spalten normalisieren
        df_ecb.columns = [c.lower() for c in df_ecb.columns[:-1]] + ['rate']
        df_ecb = df_ecb[['currency', 'time_period', 'rate']]
        df_ecb.rename(columns={'time_period': 'year'}, inplace=True)
        df_ecb['year'] = df_ecb['year'].astype(int)
        df_ecb = df_ecb[df_ecb['year'] >= start_year]
        print(f"EZB lieferte Daten für {len(df_ecb['currency'].unique())} Währungen.")
        
    except Exception as e:
        print(f"Hinweis: EZB-Abruf fehlgeschlagen ({e}).")
        df_ecb = pd.DataFrame(columns=['currency', 'year', 'rate'])

    # 2. Lücken identifizieren (Währung + Jahr)
    # Wir erstellen eine Liste aller gewünschten Kombinationen
    years = list(range(start_year, 2026)) # Bis zum aktuellen/geplanten Jahr
    yf_list = []

    for curr in currencies:
        # Welche Jahre fehlen für diese Währung bei der EZB?
        existing_years = df_ecb[df_ecb['currency'] == curr]['year'].tolist()
        missing_years = [y for y in years if y not in existing_years]
        
        if missing_years:
            print(f"-> {curr}: Ergänze {len(missing_years)} Jahre via Yahoo Finance...")
            ticker = f"EUR{curr}=X"
            # Wir laden nur ab dem ersten fehlenden Jahr, um Zeit zu sparen
            fetch_start = min(missing_years)
            data = yf.download(ticker, start=f"{fetch_start}-01-01", progress=False)
            
            if not data.empty:
                if isinstance(data.columns, pd.MultiIndex):
                    data.columns = data.columns.get_level_values(0)
                
                col_name = 'Adj Close' if 'Adj Close' in data.columns else 'Close'
                
                # Jahresdurchschnitt bilden
                annual = data[col_name].resample('YE').mean().reset_index()
                annual.columns = ['year', 'rate']
                annual['year'] = annual['year'].dt.year
                annual['currency'] = curr
                
                # Nur die Jahre behalten, die der EZB wirklich fehlten
                annual = annual[annual['year'].isin(missing_years)]
                yf_list.append(annual)

    # 3. Daten zusammenführen
    if yf_list:
        df_yf = pd.concat(yf_list, ignore_index=True)
        df_final = pd.concat([df_ecb, df_yf], ignore_index=True)
    else:
        df_final = df_ecb

    # Finales Aufräumen
    df_final['rate'] = pd.to_numeric(df_final['rate'], errors='coerce')
    df_final = df_final.dropna(subset=['rate'])
    
    # Letzter Check gegen Dubletten (Sicherheitshalber)
    df_final = df_final.drop_duplicates(subset=['currency', 'year'], keep='first')
    
    return df_final.sort_values(['currency', 'year']).reset_index(drop=True)

# Ausführung
my_currencies = [
    "AUD", "CAD", "CLP", "COP", "CRC", "CZK", "DKK", "HUF", "ISK", 
    "ILS", "JPY", "KRW", "MXN", "NZD", "NOK", "PLN", "SEK", "CHF", 
    "TRY", "GBP", "USD", "BGN", "RON"
]

fx_annual_df = get_fx_data(my_currencies)

Request class will be removed in v3.0; use Client(…)


Rufe Daten von der EZB ab...
EZB lieferte Daten für 20 Währungen.
-> CLP: Ergänze 27 Jahre via Yahoo Finance...


C:\Users\JacobStank\AppData\Local\Temp\ipykernel_31368\3705055648.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=f"{fetch_start}-01-01", progress=False)


-> COP: Ergänze 27 Jahre via Yahoo Finance...


C:\Users\JacobStank\AppData\Local\Temp\ipykernel_31368\3705055648.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=f"{fetch_start}-01-01", progress=False)


...

C:\Users\JacobStank\AppData\Local\Temp\ipykernel_31368\3705055648.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=f"{fetch_start}-01-01", progress=False)


-> ILS: Ergänze 1 Jahre via Yahoo Finance...


C:\Users\JacobStank\AppData\Local\Temp\ipykernel_31368\3705055648.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=f"{fetch_start}-01-01", progress=False)


-> BGN: Ergänze 1 Jahre via Yahoo Finance...


C:\Users\JacobStank\AppData\Local\Temp\ipykernel_31368\3705055648.py:37: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=f"{fetch_start}-01-01", progress=False)


In [17]:
fx_annual_df.to_csv("fx_annual.csv", index=False, sep=';', decimal=',')

### OECD Annual Salary Average Download not possible via API, downloading as file

In [10]:
oecd_salaries_df = pd.read_csv("OECD Wages Data.csv", sep=',', decimal='.')


oecd_salaries_df = oecd_salaries_df.rename(columns={"TIME_PERIOD": "Year", "Reference area": "Geopolitical entity (reporting)",})
oecd_salaries_df["Country Code"] = oecd_salaries_df["Geopolitical entity (reporting)"].map(country_map)
oecd_salaries_df["YearCountryKey"] = oecd_salaries_df["Year"].astype(str) + oecd_salaries_df["Country Code"]
oecd_salaries_df = oecd_salaries_df[oecd_salaries_df["Price base"] != "Constant prices"]


oecd_salaries_df["Year"] = (
    oecd_salaries_df["Year"]
        .astype(str)
        .str.extract(r"(\d{4})")      # Vierstellige Jahreszahl holen
        .astype(float)
        .astype("Int64")
)

oecd_salaries_df = oecd_salaries_df[oecd_salaries_df["Year"] >= 1999]

print(oecd_salaries_df.head())

     STRUCTURE                               STRUCTURE_ID  \
1290  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)   
1291  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)   
1292  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)   
1293  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)   
1294  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)   

            STRUCTURE_NAME ACTION REF_AREA Geopolitical entity (reporting)  \
1290  Average annual wages      I      AUT                         Austria   
1291  Average annual wages      I      AUT                         Austria   
1292  Average annual wages      I      AUT                         Austria   
1293  Average annual wages      I      AUT                         Austria   
1294  Average annual wages      I      AUT                         Austria   

     MEASURE Measure UNIT_MEASURE Unit of measure  ... BASE_PER Base period  \
1290      WG   Wages          EUR            Euro  ...      NaN         NaN   
1291  

### Bringing together Exchange Rate and OECD Data to convert currency in EUR for better comparison

In [11]:
# 1. Merge durchführen und Ergebnis direkt wieder in oecd_salaries_df speichern
oecd_salaries_df = pd.merge(
    oecd_salaries_df, 
    fx_annual_df, 
    left_on=["Year", "UNIT_MEASURE"], 
    right_on=["year", "currency"], 
    how="left"
)

# 2. Umrechnung berechnen
oecd_salaries_df["OBS_VALUE_EUR"] = oecd_salaries_df["OBS_VALUE"] / oecd_salaries_df["rate"]

# 3. Euro-Werte (1:1) korrigieren, falls kein Wechselkurs in fx_annual_df war
oecd_salaries_df.loc[oecd_salaries_df["UNIT_MEASURE"] == "EUR", "OBS_VALUE_EUR"] = oecd_salaries_df["OBS_VALUE"]

# 4. Nur die Hilfsspalten vom Merge löschen, deine ursprünglichen Spalten bleiben!
oecd_salaries_df.drop(columns=["year", "currency", "rate"], inplace=True, errors="ignore")

# Kurzer Check
print(f"Spalten im DataFrame: {oecd_salaries_df.columns.tolist()}")
print(oecd_salaries_df.head())

Spalten im DataFrame: ['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA', 'Geopolitical entity (reporting)', 'MEASURE', 'Measure', 'UNIT_MEASURE', 'Unit of measure', 'PAY_PERIOD', 'Pay period', 'PRICE_BASE', 'Price base', 'AGGREGATION_OPERATION', 'Aggregation operation', 'SEX', 'Sex', 'Year', 'Time period', 'OBS_VALUE', 'Observation value', 'BASE_PER', 'Base period', 'OBS_STATUS', 'Observation status', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals', 'Country Code', 'YearCountryKey', 'OBS_VALUE_EUR']
  STRUCTURE                               STRUCTURE_ID        STRUCTURE_NAME  \
0  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)  Average annual wages   
1  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)  Average annual wages   
2  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)  Average annual wages   
3  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)  Average annual wages   
4  DATAFLOW  OECD.ELS.SAE:DSD_EARNINGS@AV_AN_WAGE(1.0)  Average annua

In [20]:
oecd_salaries_df.to_csv("Test.csv", index=False, sep=';', decimal=',')

## Download Investments as Percentage to GDP (Yearly, Categorical)

In [12]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tec00132?format=JSON&unit=PC&indic=INV_BSN&indic=INV_GOV&indic=INV_HH&indic=INV_TOT&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
investments_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
investments_df['value'] = pd.to_numeric(investments_df['value'], errors='coerce')

# Data Cleaning and Transformation
investments_df = investments_df.dropna(subset=["value"])
investments_df = investments_df.rename(columns={"value": "Investment (in Percent of GDP)", "Time": "Year"})
investments_df = investments_df.drop(columns=["Time frequency", "Unit of measure"])
investments_df["Country Code"] = investments_df["Geopolitical entity (reporting)"].map(country_map)
investments_df["YearCountryKey"] = investments_df["Year"].astype(str) + investments_df["Country Code"]

print(investments_df.head())

          Indicator            Geopolitical entity (reporting)  Year  \
0  Total investment  European Union - 27 countries (from 2020)  2013   
1  Total investment  European Union - 27 countries (from 2020)  2014   
2  Total investment  European Union - 27 countries (from 2020)  2015   
3  Total investment  European Union - 27 countries (from 2020)  2016   
4  Total investment  European Union - 27 countries (from 2020)  2017   

   Investment (in Percent of GDP) Country Code YearCountryKey  
0                           19.86    EU27-2020  2013EU27-2020  
1                           19.94    EU27-2020  2014EU27-2020  
2                           20.37    EU27-2020  2015EU27-2020  
3                           20.54    EU27-2020  2016EU27-2020  
4                           20.82    EU27-2020  2017EU27-2020  


In [22]:
investments_df.to_csv("Investments.csv", index=False, sep=';', decimal=',')

## Download Government Spending

### Government Spending per National Account Category (Yearly, Categorical, Included in Eco)

In [34]:
# ----------------------------------
# Parameter
# ----------------------------------
BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/gov_10a_exp"

# Fixe Parameter
FIXED_PARAMS = {
    "format": "JSON",
    "unit": "MIO_EUR",
    "lang": "EN"
}

# Sektoren (wie in deiner Query)
# SECTOR_VALUES = ["S1311", "S1312", "S1313", "S1314"]
SECTOR_VALUES = ["S13"]

# Alle COFOG99-Werte aus deiner Query
#COFOG_VALUES = [
#    "GF01","GF0101","GF0102","GF0103","GF0104","GF0105","GF0106","GF0107","GF0108",
#    "GF02","GF0201","GF0202","GF0203","GF0204","GF0205",
#    "GF03","GF0301","GF0302","GF0303","GF0304","GF0305","GF0306",
#    "GF04","GF0401","GF0402","GF0403","GF0404","GF0405","GF0406","GF0407","GF0408","GF0409",
#    "GF05","GF0501","GF0502","GF0503","GF0504","GF0505","GF0506",
#    "GF06","GF0601","GF0602","GF0603","GF0604","GF0605","GF0606",
#    "GF07","GF0701","GF0702","GF0703","GF0704","GF0705","GF0706",
#    "GF08","GF0801","GF0802","GF0803","GF0804","GF0805","GF0806",
#    "GF09","GF0901","GF0902","GF0903","GF0904","GF0905","GF0906","GF0907","GF0908",
#    "GF10","GF1001","GF1002","GF1003","GF1004","GF1005","GF1006","GF1007","GF1008","GF1009"
#]

COFOG_VALUES = [
    "GF01",
    "GF02",
    "GF03",
    "GF04",
    "GF05",
    "GF06",
    "GF07",
    "GF08",
    "GF09",
    "GF10"
]

# NA_ITEM-Werte
#NA_ITEM_VALUES = [
#    "D1","D29_D5_D8","D3","D4","D4_S1311","D4_S1312","D4_S1313","D4_S1314",
#    "D62","D62_D632","D632","D7","D7_S1311","D7_S1312","D7_S1313","D7_S1314",
#    "D9","D9_S1311","D9_S1312","D9_S1313","D9_S1314","D92",
#    "NP","OP5ANP",
#    "P2","P2_D29_D5_D8","P3","P31","P32","P5","P51G",
#    "TE"
#]

# Ich dachte, die anderen NA-ITEM Werte wären sinnvoll, doch sie scheinen sich zu überschneiden. Aus Zeitgründen beschränke ich mich nun auf "TE" (Total General Government Expenditure)
NA_ITEM_VALUES = ["TE"]

# Batchgrößen – beliebig anpassbar
COFOG_BATCH_SIZE = 15
NA_ITEM_BATCH_SIZE = 12
SECTOR_BATCH_SIZE = 2   # nur 4 Sektoren → 2er-Batches sauber genug

# Retry + Timeout
REQUEST_TIMEOUT = 60
MAX_RETRIES = 3
RETRY_SLEEP = 2

# ----------------------------------
# Hilfsfunktion: batching
# ----------------------------------
def batched(iterable, n):
    it = iter(iterable)
    while True:
        batch = list(islice(it, n))
        if not batch:
            return
        yield batch

# ----------------------------------
# Parameter bauen
# ----------------------------------
def build_params(sector_list, cofog_list, na_item_list):
    params = []
    # fixe Parameter
    for k, v in FIXED_PARAMS.items():
        params.append((k, v))
    # sector
    for v in sector_list:
        params.append(("sector", v))
    # cofog99
    for v in cofog_list:
        params.append(("cofog99", v))
    # na_item
    for v in na_item_list:
        params.append(("na_item", v))
    return params

# ----------------------------------
# Batch-Fetch
# ----------------------------------
def fetch_batch(sector_batch, cofog_batch, na_item_batch, session=None):
    sess = session or requests.Session()
    params = build_params(sector_batch, cofog_batch, na_item_batch)
    last_exc = None
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = sess.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT)
            resp.raise_for_status()
            json_string = json.dumps(resp.json())
            dataset = pyjstat.Dataset.read(json_string)
            df = dataset.write('dataframe')
            return df
        except Exception as e:
            last_exc = e
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP)
            else:
                raise last_exc

# ----------------------------------
# Hauptlogik
# ----------------------------------
all_parts = []

with requests.Session() as session:
    for sector_batch in batched(SECTOR_VALUES, SECTOR_BATCH_SIZE):
        for cofog_batch in batched(COFOG_VALUES, COFOG_BATCH_SIZE):
            for na_item_batch in batched(NA_ITEM_VALUES, NA_ITEM_BATCH_SIZE):

                df_part = fetch_batch(sector_batch, cofog_batch, na_item_batch, session=session)

                if df_part is not None and not df_part.empty:
                    all_parts.append(df_part)

if not all_parts:
    raise RuntimeError("Keine Daten erhalten – prüfe Filter/Parameter.")

gov_df = pd.concat(all_parts, ignore_index=True)

# Converting 'value' to numeric, coercing errors to NaN
gov_df['value'] = pd.to_numeric(gov_df['value'], errors='coerce')

# Duplikate entfernen
possible_keys = [c for c in ["time", "geo", "sector", "cofog99", "na_item", "unit"] if c in gov_df.columns]
if "value" in gov_df.columns:
    gov_df = gov_df.drop_duplicates(subset=possible_keys + ["value"])
else:
    gov_df = gov_df.drop_duplicates(subset=possible_keys)

# Data Cleaning and Transformation
gov_df = gov_df.dropna(subset=["value"])
gov_df = gov_df.rename(columns={"value": "Government Spending (Million EUR)", "Time": "Year"})
gov_df = gov_df.drop(columns=["Time frequency", "Unit of measure"])
gov_df["Country Code"] = gov_df["Geopolitical entity (reporting)"].map(country_map)
gov_df["YearCountryKey"] = gov_df["Year"].astype(str) + gov_df["Country Code"]

print(gov_df.head())
print("Form:", gov_df.shape)

               Sector  \
5  General government   
6  General government   
7  General government   
8  General government   
9  General government   

  Classification of the functions of government (COFOG 1999)  \
5                            General public services           
6                            General public services           
7                            General public services           
8                            General public services           
9                            General public services           

  National accounts indicator (ESA 2010)  \
5   Total general government expenditure   
6   Total general government expenditure   
7   Total general government expenditure   
8   Total general government expenditure   
9   Total general government expenditure   

             Geopolitical entity (reporting)  Year  \
5  European Union - 27 countries (from 2020)  1995   
6  European Union - 27 countries (from 2020)  1996   
7  European Union - 27 countries (fro

In [35]:
gov_df.to_csv("Government_Spending.csv", index=False, sep=';', decimal=',')

## Download Consumption (in percentages) (Yearly Categorical, cannot be included in Eco)

In [15]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/tec00134?format=JSON&unit=PC_TOT&coicop=CP01&coicop=CP02&coicop=CP03&coicop=CP04&coicop=CP05&coicop=CP06&coicop=CP07&coicop=CP08&coicop=CP09&coicop=CP10&coicop=CP11&coicop=CP12&coicop=TOTAL&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
consumption_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
consumption_df['value'] = pd.to_numeric(consumption_df['value'], errors='coerce')

# Data Cleaning and Transformation
consumption_df = consumption_df.dropna(subset=["value"])
consumption_df = consumption_df.rename(columns={"value": "Consumption (in Percent of Total)", "Time": "Year"})
consumption_df = consumption_df.drop(columns=["Time frequency", "Unit of measure"])
consumption_df["Country Code"] = consumption_df["Geopolitical entity (reporting)"].map(country_map)
consumption_df["YearCountryKey"] = consumption_df["Year"].astype(str) + consumption_df["Country Code"]

print(consumption_df.head())

  Classification of individual consumption by purpose (COICOP)  \
5                   Food and non-alcoholic beverages             
6                   Food and non-alcoholic beverages             
7                   Food and non-alcoholic beverages             
8                   Food and non-alcoholic beverages             
9                   Food and non-alcoholic beverages             

             Geopolitical entity (reporting)  Year  \
5  European Union - 27 countries (from 2020)  1995   
6  European Union - 27 countries (from 2020)  1996   
7  European Union - 27 countries (from 2020)  1997   
8  European Union - 27 countries (from 2020)  1998   
9  European Union - 27 countries (from 2020)  1999   

   Consumption (in Percent of Total) Country Code YearCountryKey  
5                               14.6    EU27-2020  1995EU27-2020  
6                               14.4    EU27-2020  1996EU27-2020  
7                               14.2    EU27-2020  1997EU27-2020  
8         

In [26]:
consumption_df.to_csv("Consumption.csv", index=False, sep=';', decimal=',')

## Gross domestic product (GDP) and main components (output, expenditure and income) (Yearly, Categorical, Percentage, cannot be included in Eco)
##### It actually can be included in Eco, there are further categories which can show the GDP components. Seeing this while bing in the Power BI Analysis. Fixing to include data.

In [16]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_gdp?format=JSON&unit=CLV_I20&unit=CLV20_MEUR&unit=CP_MPPS_EU27_2020&unit=PC_GDP&unit=PD20_EUR&na_item=B1G&na_item=B1GQ&na_item=B2A3G&na_item=D1&na_item=D11&na_item=D12&na_item=D2&na_item=D21&na_item=D21X31&na_item=D2X3&na_item=D3&na_item=D31&na_item=P3&na_item=P3_P5&na_item=P3_P6&na_item=P3_S13&na_item=P31_S13&na_item=P31_S14&na_item=P31_S14_S15&na_item=P31_S15&na_item=P32_S13&na_item=P41&na_item=P51G&na_item=P52&na_item=P52_P53&na_item=P53&na_item=P5G&na_item=P6&na_item=P61&na_item=P61X71&na_item=P62&na_item=P62X72&na_item=P6X7&na_item=P7&na_item=P71&na_item=P72&na_item=YA0&na_item=YA1&na_item=YA2&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
gdp_components_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
gdp_components_df['value'] = pd.to_numeric(gdp_components_df['value'], errors='coerce')

# Data Cleaning and Transformation1
gdp_components_df = gdp_components_df.dropna(subset=["value"])

## Pivoting for Unit of measure
gdp_components_df = gdp_components_df.pivot(
    index=[col for col in gdp_components_df.columns if col not in ["Unit of measure", "value"]],
    columns="Unit of measure",
    values="value"
).reset_index()

# Data Cleaning and Transformation2
gdp_components_df = gdp_components_df.rename(columns={"Time": "Year"})
gdp_components_df = gdp_components_df.drop(columns=["Time frequency"])
gdp_components_df["Country Code"] = gdp_components_df["Geopolitical entity (reporting)"].map(country_map)
gdp_components_df["YearCountryKey"] = gdp_components_df["Year"].astype(str) + gdp_components_df["Country Code"]

print(gdp_components_df.head())

Unit of measure    National accounts indicator (ESA 2010)  \
0                Acquisitions less disposals of valuables   
1                Acquisitions less disposals of valuables   
2                Acquisitions less disposals of valuables   
3                Acquisitions less disposals of valuables   
4                Acquisitions less disposals of valuables   

Unit of measure Geopolitical entity (reporting)  Year  \
0                                       Austria  1995   
1                                       Austria  1996   
2                                       Austria  1997   
3                                       Austria  1998   
4                                       Austria  1999   

Unit of measure  Chain linked volumes (2020), million euro  \
0                                                      NaN   
1                                                      NaN   
2                                                      NaN   
3                                         

In [28]:
gdp_components_df.to_csv("GDP_Components.csv", index=False, sep=';', decimal=',')

## Download Labor

### Hours Worked (Yearly, Categorical, Included in Eco)

In [17]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_a10_e?format=JSON&unit=THS_HW&unit=THS_JOB&unit=THS_PER&nace_r2=A&nace_r2=B-E&nace_r2=C&nace_r2=F&nace_r2=G-I&nace_r2=J&nace_r2=K&nace_r2=L&nace_r2=M_N&nace_r2=O-Q&nace_r2=R-U&na_item=SAL_DC&na_item=SELF_DC&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
labor_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
labor_df['value'] = pd.to_numeric(labor_df['value'], errors='coerce')

labor_df = labor_df.dropna(subset=["value"])

labor_df = labor_df.pivot(
    index=[col for col in labor_df.columns if col not in ["Unit of measure", "value"]],
    columns="Unit of measure",
    values="value"
).reset_index()

# Data Cleaning and Transformation
labor_df = labor_df.rename(columns={"Time": "Year"})
labor_df = labor_df.drop(columns=["Time frequency"])
labor_df["Country Code"] = labor_df["Geopolitical entity (reporting)"].map(country_map)
labor_df["YearCountryKey"] = labor_df["Year"].astype(str) + labor_df["Country Code"]
labor_df["Thousand hours worked"] = labor_df["Thousand hours worked"].replace(0, pd.NA)
labor_df["Thousand jobs"] = labor_df["Thousand jobs"].replace(0, pd.NA)
labor_df["Thousand persons"] = labor_df["Thousand persons"].replace(0, pd.NA)

# Dropping NA Values
cols_to_check = ["Thousand hours worked", "Thousand jobs", "Thousand persons"]
labor_df = labor_df.dropna(subset=cols_to_check, how='all')

print(labor_df.head())

Unit of measure Statistical classification of economic activities in the European Community (NACE Rev. 2)  \
0                                Agriculture, forestry and fishing                                          
1                                Agriculture, forestry and fishing                                          
2                                Agriculture, forestry and fishing                                          
3                                Agriculture, forestry and fishing                                          
4                                Agriculture, forestry and fishing                                          

Unit of measure National accounts indicator (ESA 2010)  \
0                           Employees domestic concept   
1                           Employees domestic concept   
2                           Employees domestic concept   
3                           Employees domestic concept   
4                           Employees domestic concept 

In [30]:
labor_df.to_csv("Labor.csv", index=False, sep=';', decimal=',')

### Job vacancies published (good) (At first, this Dataset was to be ignored for time reasons. Yet, it results to be useful so it is added later seperately to the Power BI File)

In [18]:
# Gemenis second try:

# 1. Fetch Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/jvs_q_nace2?format=JSON&s_adj=NSA&nace_r2=A&nace_r2=B&nace_r2=C&nace_r2=D&nace_r2=E&nace_r2=F&nace_r2=G&nace_r2=H&nace_r2=I&nace_r2=J&nace_r2=K&nace_r2=L&nace_r2=M&nace_r2=N&nace_r2=O&nace_r2=P&nace_r2=Q&nace_r2=R&nace_r2=S&sizeclas=TOTAL&indic_em=JOBOCC&indic_em=JOBVAC&indic_em=JVR&indic_em=JVRCH_Q&lang=EN"

response = requests.get(url)
response.raise_for_status()
json_string = json.dumps(response.json())
dataset = pyjstat.Dataset.read(json_string)
df = dataset.write('dataframe')

# 2. Pre-Pivot Cleaning
# Ensure the value column is numeric; 'coerce' turns non-numeric flags into NaN
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# 3. Pivot the Data
# This separates the different indicators (Job vacancies, Job occupied, etc.) into columns
job_vacancies_df = df.pivot(
    index=[col for col in df.columns if col not in ["Employment indicator", "value"]],
    columns="Employment indicator",
    values="value"
).reset_index()

# Replacing value 0 with NaN
# It is not plausible that a country has 0 open Job vacancies.
job_vacancies_df["Job vacancies"] = job_vacancies_df["Job vacancies"].replace(0, pd.NA)

# 4. Feature Engineering
job_vacancies_df["Year"] = job_vacancies_df["Time"].str.slice(0, 4).astype(int)
job_vacancies_df["Quarter"] = job_vacancies_df["Time"].str.slice(6, 7).astype(int)

# Use .get() for the map to avoid errors if a country code is missing in your map
# (Assuming country_map is defined in your environment)
job_vacancies_df["Country Code"] = job_vacancies_df["Geopolitical entity (reporting)"].map(country_map)

# 5. Smart Fill: Filling missing Quarters with Yearly Averages
group_keys = [
    "Statistical classification of economic activities in the European Community (NACE Rev. 2)",
    "Country Code",
    "Year"
]

# Calculate stats per group
# transform('mean') handles NaNs automatically (it ignores them in the average)
job_vacancies_df["Yearly_Mean"] = job_vacancies_df.groupby(group_keys)["Job vacancies"].transform("mean")
job_vacancies_df["Quarters_Present"] = job_vacancies_df.groupby(group_keys)["Job vacancies"].transform("count")

# Logic: Fill only if original value is NaN and we have at least one valid quarter to average from
# If you only want to fill for "complete" years, change > 0 to == 4
mask = job_vacancies_df["Job vacancies"].isna() & (job_vacancies_df["Quarters_Present"] == 4)
job_vacancies_df.loc[mask, "Job vacancies"] = job_vacancies_df.loc[mask, "Yearly_Mean"]

# 6. Final Formatting
job_vacancies_df["YearCountryKey"] = job_vacancies_df["Year"].astype(str) + job_vacancies_df["Country Code"].astype(str)

# Drop helper columns and unnecessary dimensions
job_vacancies_df = job_vacancies_df.drop(columns=["Time frequency", "Seasonal adjustment", "Yearly_Mean", "Quarters_Present"])

# Round the filled values to integers if appropriate for vacancies
# job_vacancies_df["Job vacancies"] = job_vacancies_df["Job vacancies"].round(0)

print("Process Complete. High-value anomaly check:")
print(job_vacancies_df["Job vacancies"].describe())

# It took a very long time to get to this code. So I will stop here and only proceed with the Job vacancies

Process Complete. High-value anomaly check:
count     33542.0
unique     9868.0
top        2000.0
freq        132.0
Name: Job vacancies, dtype: float64


In [32]:
job_vacancies_df.to_csv("Job_Vacancies.csv", index=False, sep=';', decimal=',')

## Population Data (good)

In [19]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/demo_pjan?format=JSON&unit=NR&age=TOTAL&sex=F&sex=M&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
population_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
population_df['value'] = pd.to_numeric(population_df['value'], errors='coerce')

# Dropping "Germany" because it is not including the former GDR
population_df = population_df[population_df["Geopolitical entity (reporting)"] != "Germany"]

# Replacing the Germany value that includes former GDR with "Germany" for consistency throughout the project
population_df["Geopolitical entity (reporting)"] = (
    population_df["Geopolitical entity (reporting)"]
        .replace("Germany including former GDR", "Germany")
        .str.strip()
)

# Data Cleaning and Transformation
population_df = population_df.dropna(subset=["value"])
population_df = population_df.rename(columns={"value": "Population", "Time": "Year"})
population_df = population_df.drop(columns=["Time frequency", "Unit of measure", "Age class"])
population_df["Country Code"] = population_df["Geopolitical entity (reporting)"].map(country_map)
population_df["YearCountryKey"] = population_df["Year"].astype(str) + population_df["Country Code"]

# Due to "to_numeric" each value now ends on ".0" which confuses microsoft programs
# Changing to Int to avoid this issue
population_df["Population"] = population_df["Population"].astype('int64')

print(population_df.head())
print(population_df["Population"].describe())

      Sex            Geopolitical entity (reporting)  Year  Population  \
25  Males  European Union - 27 countries (from 2020)  1985   200118494   
30  Males  European Union - 27 countries (from 2020)  1990   203365156   
31  Males  European Union - 27 countries (from 2020)  1991   204135430   
32  Males  European Union - 27 countries (from 2020)  1992   204599131   
33  Males  European Union - 27 countries (from 2020)  1993   205413335   

   Country Code YearCountryKey  
25    EU27-2020  1985EU27-2020  
30    EU27-2020  1990EU27-2020  
31    EU27-2020  1991EU27-2020  
32    EU27-2020  1992EU27-2020  
33    EU27-2020  1993EU27-2020  
count    5.326000e+03
mean     2.282005e+07
std      5.714438e+07
min      7.932000e+03
25%      1.526326e+06
50%      4.120751e+06
75%      1.164007e+07
max      2.647706e+08
Name: Population, dtype: float64


In [34]:
population_df.to_csv("Population.csv", index=False, sep=';', decimal=',')

## Download BIP-Deflator (2015 = 100)

In [20]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_gdp?format=JSON&unit=PD15_NAC&na_item=B1GQ&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
deflator2015_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
deflator2015_df['value'] = pd.to_numeric(deflator2015_df['value'], errors='coerce')

# Data Cleaning and Transformation
deflator2015_df = deflator2015_df.dropna(subset=["value"])
deflator2015_df = deflator2015_df.rename(columns={"value": "GDP Deflator 2015", "Time": "Year"})
deflator2015_df["Country Code"] = deflator2015_df["Geopolitical entity (reporting)"].map(country_map)
deflator2015_df["YearCountryKey"] = deflator2015_df["Year"].astype(str) + deflator2015_df["Country Code"]

# Due to "to_numeric" each value now ends on ".0" which confuses microsoft programs
# Changing to Int to avoid this issue
deflator2015_df["Year"] = deflator2015_df["Year"].astype('int64')

print(deflator2015_df.head())
print(deflator2015_df["GDP Deflator 2015"].describe())

   Time frequency                                    Unit of measure  \
20         Annual  Price index (implicit deflator), 2015=100, nat...   
21         Annual  Price index (implicit deflator), 2015=100, nat...   
22         Annual  Price index (implicit deflator), 2015=100, nat...   
23         Annual  Price index (implicit deflator), 2015=100, nat...   
24         Annual  Price index (implicit deflator), 2015=100, nat...   

     National accounts indicator (ESA 2010)  \
20  Gross domestic product at market prices   
21  Gross domestic product at market prices   
22  Gross domestic product at market prices   
23  Gross domestic product at market prices   
24  Gross domestic product at market prices   

              Geopolitical entity (reporting)  Year  GDP Deflator 2015  \
20  European Union - 27 countries (from 2020)  1995             71.321   
21  European Union - 27 countries (from 2020)  1996             73.357   
22  European Union - 27 countries (from 2020)  1997           

In [36]:
deflator2015_df.to_csv("Deflator2015.csv", index=False, sep=';', decimal=',')

## Creating Economic DataFrame

In [98]:
mm_ir_yy_df["Year"] = (mm_ir_yy_df["Year"].astype("Int64"))
gdp_pc_df["Year"] = (gdp_pc_df["Year"].astype("Int64"))

economic_data_df = pd.merge(
    mm_ir_yy_df, 
    gdp_pc_df, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

economic_data_df = economic_data_df.drop(columns=['National accounts indicator (ESA 2010)'])

print(economic_data_df.head())

  Geopolitical entity (reporting)  Year  Interest Rate Country Code  \
0                         Denmark  1970        9.00000           DK   
1                  United Kingdom  1970        8.05740           UK   
2                         Denmark  1971        7.62329           DK   
3                  United Kingdom  1971        6.19932           UK   
4                         Denmark  1972        7.25137           DK   

  YearCountryKey  Real GDP (2015 = 100) (in EUR) per Capita  
0         1970DK                                        NaN  
1         1970UK                                        NaN  
2         1971DK                                        NaN  
3         1971UK                                        NaN  
4         1972DK                                        NaN  


In [99]:
# Groubing by Country and Year because the data is devided into goods and services.
imports_total = imports_percgdp_df.groupby([
    'YearCountryKey', 
    'Year', 
    'Country Code', 
    'Geopolitical entity (reporting)'
])['Imports as Percentage of GDP'].sum().reset_index()

imports_total["Year"] = (imports_total["Year"].astype("Int64"))

# Expanding the general economic data frame
economic_data_df = pd.merge(
    economic_data_df, 
    imports_total, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'],
    how='outer'
)

imports = imports_percgdp_df.pivot(
    index=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'],
    columns="National accounts indicator (ESA 2010)",
    values="Imports as Percentage of GDP"
).reset_index()

imports = imports.rename(columns={"Imports of goods": "Imports of Goods as Percentage of GDP", "Imports of services": "Imports of Services as Percentage of GDP"})

imports["Year"] = (imports["Year"].astype("Int64"))

economic_data_df = pd.merge(
    economic_data_df, 
    imports, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [100]:
# Filtering for total values
fva_total_nace = fva_df[
    fva_df['Statistical classification of economic activities in the European Community (NACE Rev. 2)'] == 'Total - all NACE activities'
].copy()

# Aggregating FVA by Country and Year
fva_aggregated = fva_total_nace.groupby([
    'YearCountryKey', 
    'Geopolitical entity (reporting)', 
    'Year',
    'Country Code'
])['Foreign Value Added (Thousand EUR)'].sum().reset_index()

fva_aggregated["Year"] = (fva_aggregated["Year"].astype("Int64"))

# Expanding the economic df with the FVA data
economic_data_df = pd.merge(
    economic_data_df, 
    fva_aggregated, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [101]:
hicp_df["Year"] = (hicp_df["Year"].astype("Int64"))

# Expanding the economic df with the FVA data
economic_data_df = pd.merge(
    economic_data_df, 
    hicp_df, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [102]:
economic_data_df["Year"] = (
    economic_data_df["Year"]
        .astype("Int64")
)

oecd_salaries_filtered = oecd_salaries_df[oecd_salaries_df['Price base'] == 'Current prices']


oecd_salaries_filtered = oecd_salaries_filtered[
    ['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)', 'OBS_VALUE_EUR']
]

oecd_salaries_filtered = oecd_salaries_filtered.rename(columns={"OBS_VALUE_EUR": "Average Annual Wages in Euros OECD (Current Prices 2024)"})

economic_data_df = pd.merge(
    economic_data_df, 
    oecd_salaries_filtered, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [103]:
investments_df["Year"] = (investments_df["Year"].astype("Int64"))

investment_filtered = investments_df[investments_df['Indicator'] == 'Total investment']

economic_data_df = pd.merge(
    economic_data_df, 
    investment_filtered, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [104]:
gov_df["Year"] = (gov_df["Year"].astype("Int64"))

gov_filtered = gov_df[gov_df['Sector'] == 'Central government']
gov_filtered = gov_df[gov_df['National accounts indicator (ESA 2010)'] == 'Total general government expenditure']

gov_aggregated = gov_filtered.groupby([
    'YearCountryKey', 
    'Geopolitical entity (reporting)', 
    'Year',
    'Country Code'
])['Government Spending (Million EUR)'].sum().reset_index()

economic_data_df = pd.merge(
    economic_data_df, 
    gov_aggregated, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [105]:
labor_df["Year"] = (labor_df["Year"].astype("Int64"))

# Adding Hours worked
labor_filtered = labor_df[
    ['Statistical classification of economic activities in the European Community (NACE Rev. 2)', 'National accounts indicator (ESA 2010)', 'YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)', 'Thousand hours worked']
]

labor_aggregated = labor_filtered.groupby([
    'YearCountryKey', 
    'Geopolitical entity (reporting)', 
    'Year',
    'Country Code'
])['Thousand hours worked'].sum().reset_index()

economic_data_df = pd.merge(
    economic_data_df, 
    labor_aggregated, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

# Adding Thousand jobs
labor_filtered = labor_df[
    ['Statistical classification of economic activities in the European Community (NACE Rev. 2)', 'National accounts indicator (ESA 2010)', 'YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)', 'Thousand jobs']
]

labor_aggregated = labor_filtered.groupby([
    'YearCountryKey', 
    'Geopolitical entity (reporting)', 
    'Year',
    'Country Code'
])['Thousand jobs'].sum().reset_index()

economic_data_df = pd.merge(
    economic_data_df, 
    labor_aggregated, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

# Adding Thousand persons
labor_filtered = labor_df[
    ['Statistical classification of economic activities in the European Community (NACE Rev. 2)', 'National accounts indicator (ESA 2010)', 'YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)', 'Thousand persons']
]

labor_aggregated = labor_filtered.groupby([
    'YearCountryKey', 
    'Geopolitical entity (reporting)', 
    'Year',
    'Country Code'
])['Thousand persons'].sum().reset_index()

economic_data_df = pd.merge(
    economic_data_df, 
    labor_aggregated, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

economic_data_df = economic_data_df.rename(columns={"Thousand persons": "Thousand Persons Employed"})

In [106]:
population_df["Year"] = (population_df["Year"].astype("Int64"))

population_aggregated = population_df.groupby([
    'YearCountryKey', 
    'Geopolitical entity (reporting)', 
    'Year',
    'Country Code'
])['Population'].sum().reset_index()

economic_data_df = pd.merge(
    economic_data_df, 
    population_aggregated, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [107]:
gdp_components_filtered = gdp_components_df.drop(columns=["Price index (implicit deflator), 2020=100, euro", "Current prices, million purchasing power standards (PPS, EU27 from 2020)", "Chain linked volumes, index 2020=100", "Chain linked volumes (2020), million euro"])

gdp_components_pivot = gdp_components_filtered.pivot(
    index=[col for col in gdp_components_filtered.columns if col not in ["National accounts indicator (ESA 2010)", "Percentage of gross domestic product (GDP)"]],
    columns="National accounts indicator (ESA 2010)",
    values="Percentage of gross domestic product (GDP)"
).reset_index()

gdp_components_selected = gdp_components_pivot[
    ['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)', 'Exports of goods', 'Exports of goods and services', 'Exports of services', 'Final consumption expenditure', 'Imports of goods', 'Imports of goods and services', 'Imports of services', 'Wages and salaries','Taxes on products', 'Taxes on production and imports', 'Subsidies', 'Subsidies on products']
]

gdp_components_selected = gdp_components_selected.rename(columns={'Exports of goods': 'Exports of Goods as Percentage of GDP', 'Exports of goods and services': 'Exports of Goods and Services as Percentage of GDP', 'Exports of services': 'Exports of Services as Percentage of GDP', 'Final consumption expenditure': 'Final Consumption Expenditure as Percentage of GDP', 'Imports of goods': 'Imports of Goods as Percentage of GDP', 'Imports of goods and services': 'Imports of Goods and Services as Percentage of GDP', 'Imports of services': 'Imports of Services as Percentage of GDP', 'Wages and salaries': 'Wages and Salaries as Percentage of GDP', 'Taxes on products': 'Taxes on Products as Percentage of GDP', 'Taxes on production and imports': 'Taxes on Production and Imports as Percentage of GDP', 'Subsidies': 'Subsidies as Percentage of GDP', 'Subsidies on products': 'Subsidies on Products as Percentage of GDP'})

gdp_components_selected["Year"] = (gdp_components_selected["Year"].astype("Int64"))

economic_data_df = pd.merge(
    economic_data_df, 
    gdp_components_selected, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

In [108]:
deflator_merge = deflator2015_df[
    ['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)', 'GDP Deflator 2015']
]

economic_data_df = pd.merge(
    economic_data_df, 
    deflator_merge, 
    on=['YearCountryKey', 'Year', 'Country Code', 'Geopolitical entity (reporting)'], 
    how='outer'
)

# Inflation will be the growth rate of the GDP deflator.
# Although the GDP deflator depends on the quatity, the formula is ptqt/pt-1qt
# Due to the vecotr characteristic, the quantities do not cancel out, however, they are held constant
# That is why the growth rate of the GDP deflator represents the change in prices
# The growth rate for indices is independent of the basis year, which is why the result is true for inflation
# Thus, Inflation can be calculated via Feature Engineering on the GDP deflator
# This step will be done in Power Query, due to seperator and data format inconsistencies in Python

In [109]:
economic_data_df = economic_data_df.drop(columns=["Indicator"])
economic_data_df = economic_data_df.rename(columns={"OBS_VALUE": "Average Annual Wages in Euros OECD (Current Prices 2024)"})

# Drop all rows in which YearCountryKey is NA
economic_data_df = economic_data_df.dropna(subset=["YearCountryKey"])

In [110]:
economic_data_df.to_csv("Economic.csv", index=False, sep=';', decimal=',')

### Apparently there are duplicates for the key variable which I see in the ongoing analysis. In this section I will check for the Key Variable to be unique - This Section has been used for Bug Fixing during the Project. It is not needed to excecute this to get to the final version

In [78]:
# Alle Zeilen herausfiltern, deren YearCountryKey mehrfach vorkommt
economic_duplicates = economic_data_df[
    economic_data_df.duplicated(subset="YearCountryKey", keep=False)
].sort_values("YearCountryKey")

print("Anzahl der doppelten YearCountryKeys:", economic_duplicates["YearCountryKey"].nunique())

Anzahl der doppelten YearCountryKeys: 14


In [79]:
economic_duplicates.to_csv("Economic Duplicates.csv", index=False, sep=';', decimal=',')

## EU Erdgas und LNG Importe

In [15]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_ti_gas?format=JSON&unit=TJ_GCV&siec=G3000&partner=AD&partner=AE&partner=AFR_OTH&partner=AL&partner=AM&partner=AME_OTH&partner=AO&partner=AR&partner=ASI_NME_OTH&partner=ASI_OTH&partner=AT&partner=AU&partner=AW&partner=AZ&partner=BA&partner=BB&partner=BD&partner=BE&partner=BG&partner=BH&partner=BJ&partner=BN&partner=BO&partner=BR&partner=BS&partner=BW&partner=BY&partner=BZ&partner=CA&partner=CD&partner=CG&partner=CH&partner=CI&partner=CL&partner=CM&partner=CN&partner=CO&partner=CR&partner=CU&partner=CV&partner=CW&partner=CY&partner=CZ&partner=DE&partner=DJ&partner=DK&partner=DO&partner=DZ&partner=EC&partner=EE&partner=EG&partner=EL&partner=ER&partner=ES&partner=ET&partner=EUR_OTH&partner=EX_SU_OTH&partner=FI&partner=FR&partner=GA&partner=GE&partner=GH&partner=GI&partner=GQ&partner=GT&partner=GW&partner=GY&partner=HK&partner=HN&partner=HR&partner=HU&partner=ID&partner=IE&partner=IL&partner=IN&partner=IQ&partner=IR&partner=IS&partner=IT&partner=JM&partner=JO&partner=JP&partner=KE&partner=KG&partner=KH&partner=KP&partner=KR&partner=KW&partner=KZ&partner=LA&partner=LB&partner=LI&partner=LK&partner=LR&partner=LT&partner=LU&partner=LV&partner=LY&partner=MA&partner=MD&partner=ME&partner=MG&partner=MH&partner=MK&partner=MM&partner=MN&partner=MR&partner=MT&partner=MU&partner=MX&partner=MY&partner=MZ&partner=NA&partner=NC&partner=NE&partner=NG&partner=NL&partner=NO&partner=NP&partner=NSP&partner=NZ&partner=OM&partner=PA&partner=PE&partner=PG&partner=PH&partner=PK&partner=PL&partner=PT&partner=QA&partner=RO&partner=RS&partner=RU&partner=SA&partner=SD&partner=SE&partner=SG&partner=SI&partner=SK&partner=SL&partner=SN&partner=SS&partner=ST&partner=SY&partner=TG&partner=TH&partner=TJ&partner=TL&partner=TM&partner=TN&partner=TR&partner=TT&partner=TW&partner=TZ&partner=UA&partner=UG&partner=UK&partner=US&partner=UY&partner=UZ&partner=VE&partner=VG&partner=VN&partner=XK&partner=YE&partner=ZA&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
natural_gas_imports_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
natural_gas_imports_df['value'] = pd.to_numeric(natural_gas_imports_df['value'], errors='coerce')

# Data Cleaning and Transformation
natural_gas_imports_df = natural_gas_imports_df.dropna(subset=["value"])
natural_gas_imports_df = natural_gas_imports_df.rename(columns={"value": "Gas Imports (in TJ)", "Time": "Year"})
natural_gas_imports_df = natural_gas_imports_df.drop(columns=["Time frequency", "Unit of measure"])
natural_gas_imports_df["Country Code"] = natural_gas_imports_df["Geopolitical entity (reporting)"].map(country_map)
natural_gas_imports_df["YearCountryKey"] = natural_gas_imports_df["Year"].astype(str) + natural_gas_imports_df["Country Code"]

print(natural_gas_imports_df.head())

  Standard international energy product classification (SIEC)  \
0                                        Natural gas            
1                                        Natural gas            
2                                        Natural gas            
3                                        Natural gas            
4                                        Natural gas            

  Geopolitical entity (partner)            Geopolitical entity (reporting)  \
0                       Belgium  European Union - 27 countries (from 2020)   
1                       Belgium  European Union - 27 countries (from 2020)   
2                       Belgium  European Union - 27 countries (from 2020)   
3                       Belgium  European Union - 27 countries (from 2020)   
4                       Belgium  European Union - 27 countries (from 2020)   

   Year  Gas Imports (in TJ) Country Code YearCountryKey  
0  1990                  0.0    EU27-2020  1990EU27-2020  
1  1991               

In [16]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_ti_gas?format=JSON&unit=TJ_GCV&siec=G3200&partner=AD&partner=AE&partner=AFR_OTH&partner=AL&partner=AM&partner=AME_OTH&partner=AO&partner=AR&partner=ASI_NME_OTH&partner=ASI_OTH&partner=AT&partner=AU&partner=AW&partner=AZ&partner=BA&partner=BB&partner=BD&partner=BE&partner=BG&partner=BH&partner=BJ&partner=BN&partner=BO&partner=BR&partner=BS&partner=BW&partner=BY&partner=BZ&partner=CA&partner=CD&partner=CG&partner=CH&partner=CI&partner=CL&partner=CM&partner=CN&partner=CO&partner=CR&partner=CU&partner=CV&partner=CW&partner=CY&partner=CZ&partner=DE&partner=DJ&partner=DK&partner=DO&partner=DZ&partner=EC&partner=EE&partner=EG&partner=EL&partner=ER&partner=ES&partner=ET&partner=EUR_OTH&partner=EX_SU_OTH&partner=FI&partner=FR&partner=GA&partner=GE&partner=GH&partner=GI&partner=GQ&partner=GT&partner=GW&partner=GY&partner=HK&partner=HN&partner=HR&partner=HU&partner=ID&partner=IE&partner=IL&partner=IN&partner=IQ&partner=IR&partner=IS&partner=IT&partner=JM&partner=JO&partner=JP&partner=KE&partner=KG&partner=KH&partner=KP&partner=KR&partner=KW&partner=KZ&partner=LA&partner=LB&partner=LI&partner=LK&partner=LR&partner=LT&partner=LU&partner=LV&partner=LY&partner=MA&partner=MD&partner=ME&partner=MG&partner=MH&partner=MK&partner=MM&partner=MN&partner=MR&partner=MT&partner=MU&partner=MX&partner=MY&partner=MZ&partner=NA&partner=NC&partner=NE&partner=NG&partner=NL&partner=NO&partner=NP&partner=NSP&partner=NZ&partner=OM&partner=PA&partner=PE&partner=PG&partner=PH&partner=PK&partner=PL&partner=PT&partner=QA&partner=RO&partner=RS&partner=RU&partner=SA&partner=SD&partner=SE&partner=SG&partner=SI&partner=SK&partner=SL&partner=SN&partner=SS&partner=ST&partner=SY&partner=TG&partner=TH&partner=TJ&partner=TL&partner=TM&partner=TN&partner=TR&partner=TT&partner=TW&partner=TZ&partner=UA&partner=UG&partner=UK&partner=US&partner=UY&partner=UZ&partner=VE&partner=VG&partner=VN&partner=XK&partner=YE&partner=ZA&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
lng_imports_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
lng_imports_df['value'] = pd.to_numeric(lng_imports_df['value'], errors='coerce')

# Data Cleaning and Transformation
lng_imports_df = lng_imports_df.dropna(subset=["value"])
lng_imports_df = lng_imports_df.rename(columns={"value": "Gas Imports (in TJ)", "Time": "Year"})
lng_imports_df = lng_imports_df.drop(columns=["Time frequency", "Unit of measure"])
lng_imports_df["Country Code"] = lng_imports_df["Geopolitical entity (reporting)"].map(country_map)
lng_imports_df["YearCountryKey"] = lng_imports_df["Year"].astype(str) + lng_imports_df["Country Code"]

print(lng_imports_df.head())

  Standard international energy product classification (SIEC)  \
0                              Liquefied natural gas            
1                              Liquefied natural gas            
2                              Liquefied natural gas            
3                              Liquefied natural gas            
4                              Liquefied natural gas            

  Geopolitical entity (partner)            Geopolitical entity (reporting)  \
0                       Belgium  European Union - 27 countries (from 2020)   
1                       Belgium  European Union - 27 countries (from 2020)   
2                       Belgium  European Union - 27 countries (from 2020)   
3                       Belgium  European Union - 27 countries (from 2020)   
4                       Belgium  European Union - 27 countries (from 2020)   

   Year  Gas Imports (in TJ) Country Code YearCountryKey  
0  1990                  0.0    EU27-2020  1990EU27-2020  
1  1991               

In [17]:
gas_imports_df = pd.concat([natural_gas_imports_df, lng_imports_df], axis=0)

In [18]:
gas_imports_df.to_csv("Natural Gas and LNG Imports.csv", index=False, sep=';', decimal=',')

## Downloading EU Cruide Oil Imports

In [21]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_ti_oil?format=JSON&unit=THS_T&siec=O4100_TOT&partner=AD&partner=AE&partner=AFR_OTH&partner=AL&partner=AM&partner=AME_OTH&partner=AO&partner=AR&partner=ASI_NME_OTH&partner=ASI_OTH&partner=AT&partner=AU&partner=AW&partner=AZ&partner=BA&partner=BB&partner=BD&partner=BE&partner=BG&partner=BH&partner=BJ&partner=BN&partner=BO&partner=BR&partner=BS&partner=BW&partner=BY&partner=BZ&partner=CA&partner=CD&partner=CG&partner=CH&partner=CI&partner=CL&partner=CM&partner=CN&partner=CO&partner=CR&partner=CU&partner=CV&partner=CW&partner=CY&partner=CZ&partner=DE&partner=DJ&partner=DK&partner=DO&partner=DZ&partner=EC&partner=EE&partner=EG&partner=EL&partner=ER&partner=ES&partner=ET&partner=EUR_OTH&partner=EX_SU_OTH&partner=FI&partner=FR&partner=GA&partner=GE&partner=GH&partner=GI&partner=GQ&partner=GT&partner=GW&partner=GY&partner=HK&partner=HN&partner=HR&partner=HU&partner=ID&partner=IE&partner=IL&partner=IN&partner=IQ&partner=IR&partner=IS&partner=IT&partner=JM&partner=JO&partner=JP&partner=KE&partner=KG&partner=KH&partner=KP&partner=KR&partner=KW&partner=KZ&partner=LA&partner=LB&partner=LI&partner=LK&partner=LR&partner=LT&partner=LU&partner=LV&partner=LY&partner=MA&partner=MD&partner=ME&partner=MG&partner=MH&partner=MK&partner=MM&partner=MN&partner=MR&partner=MT&partner=MU&partner=MX&partner=MY&partner=MZ&partner=NA&partner=NC&partner=NE&partner=NG&partner=NL&partner=NO&partner=NP&partner=NSP&partner=NZ&partner=OM&partner=PA&partner=PE&partner=PG&partner=PH&partner=PK&partner=PL&partner=PT&partner=QA&partner=RO&partner=RS&partner=RU&partner=SA&partner=SD&partner=SE&partner=SG&partner=SI&partner=SK&partner=SL&partner=SN&partner=SS&partner=ST&partner=SY&partner=TG&partner=TH&partner=TJ&partner=TL&partner=TM&partner=TN&partner=TOTAL&partner=TR&partner=TT&partner=TW&partner=TZ&partner=UA&partner=UG&partner=UK&partner=US&partner=UY&partner=UZ&partner=VE&partner=VG&partner=VN&partner=XK&partner=YE&partner=ZA&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
cruide_oil_imports_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
cruide_oil_imports_df['value'] = pd.to_numeric(cruide_oil_imports_df['value'], errors='coerce')

# Data Cleaning and Transformation
cruide_oil_imports_df = cruide_oil_imports_df.dropna(subset=["value"])
cruide_oil_imports_df = cruide_oil_imports_df.rename(columns={"value": "Cruide Oil Imports (in TSD Tonnes)", "Time": "Year"})
cruide_oil_imports_df = cruide_oil_imports_df.drop(columns=["Time frequency", "Unit of measure"])
cruide_oil_imports_df["Country Code"] = cruide_oil_imports_df["Geopolitical entity (reporting)"].map(country_map)
cruide_oil_imports_df["YearCountryKey"] = cruide_oil_imports_df["Year"].astype(str) + cruide_oil_imports_df["Country Code"]

print(cruide_oil_imports_df.head())

  Standard international energy product classification (SIEC)  \
0                                          Crude oil            
1                                          Crude oil            
2                                          Crude oil            
3                                          Crude oil            
4                                          Crude oil            

  Geopolitical entity (partner)            Geopolitical entity (reporting)  \
0                       Belgium  European Union - 27 countries (from 2020)   
1                       Belgium  European Union - 27 countries (from 2020)   
2                       Belgium  European Union - 27 countries (from 2020)   
3                       Belgium  European Union - 27 countries (from 2020)   
4                       Belgium  European Union - 27 countries (from 2020)   

   Year  Cruide Oil Imports (in TSD Tonnes) Country Code YearCountryKey  
0  1990                                 0.0    EU27-2020  1990EU27

In [20]:
cruide_oil_imports_df.to_csv("Cruide Oil Imports.csv", index=False, sep=';', decimal=',')

## Downloading Oil and Gas Prices

In [ ]:
start_date = "2010-01-01"
end_date = None

# --- Preise laden ---
brent = yf.download("BZ=F", start=start_date, end=end_date, progress=False)
ttf   = yf.download("TTF=F", start=start_date, end=end_date, progress=False)
eurusd = yf.download("EURUSD=X", start=start_date, end=end_date, progress=False)

# --- Close extrahieren ---
brent_close = brent.loc[:, ("Close", "BZ=F")]
ttf_close   = ttf.loc[:, ("Close", "TTF=F")]
eurusd_close = eurusd.loc[:, ("Close", "EURUSD=X")]

# --- DataFrame bauen ---
prices_df = pd.DataFrame({
    "Oil Brent Price (EUR per Bbl)": brent_close / eurusd_close,
    "Gas TTF Price (EUR per MWh)": ttf_close
})

prices_df = prices_df.dropna()
prices_df.index.name = "date"

print(prices_df.head())

C:\Users\JacobStank\AppData\Local\Temp\ipykernel_8304\2056958057.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  brent = yf.download("BZ=F", start=start_date, end=end_date, progress=False)
C:\Users\JacobStank\AppData\Local\Temp\ipykernel_8304\2056958057.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  ttf   = yf.download("TTF=F", start=start_date, end=end_date, progress=False)
C:\Users\JacobStank\AppData\Local\Temp\ipykernel_8304\2056958057.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  eurusd = yf.download("EURUSD=X", start=start_date, end=end_date, progress=False)


            Oil Brent Price (EUR per Bbl)  Gas TTF Price (EUR per MWh)
Date                                                                  
2017-10-23                      48.810395                    18.090000
2017-10-24                      49.603831                    17.959999
2017-10-25                      49.697957                    18.110001
2017-10-26                      50.166020                    18.070000
2017-10-27                      51.936693                    18.150000


In [36]:
prices_df.to_csv("Oil_and_Gas_Prices.csv", index=True, sep=';', decimal=',')

## Downloading Jetfuel Imports

In [39]:
# Defining Eurostats URL, here: yearly Data
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_ti_oil?format=JSON&unit=THS_T&siec=O4661&partner=AD&partner=AE&partner=AFR_OTH&partner=AL&partner=AM&partner=AME_OTH&partner=AO&partner=AR&partner=ASI_NME_OTH&partner=ASI_OTH&partner=AT&partner=AU&partner=AW&partner=AZ&partner=BA&partner=BB&partner=BD&partner=BE&partner=BG&partner=BH&partner=BJ&partner=BN&partner=BO&partner=BR&partner=BS&partner=BW&partner=BY&partner=BZ&partner=CA&partner=CD&partner=CG&partner=CH&partner=CI&partner=CL&partner=CM&partner=CN&partner=CO&partner=CR&partner=CU&partner=CV&partner=CW&partner=CY&partner=CZ&partner=DE&partner=DJ&partner=DK&partner=DO&partner=DZ&partner=EC&partner=EE&partner=EG&partner=EL&partner=ER&partner=ES&partner=ET&partner=EUR_OTH&partner=EX_SU_OTH&partner=FI&partner=FR&partner=GA&partner=GE&partner=GH&partner=GI&partner=GQ&partner=GT&partner=GW&partner=GY&partner=HK&partner=HN&partner=HR&partner=HU&partner=ID&partner=IE&partner=IL&partner=IN&partner=IQ&partner=IR&partner=IS&partner=IT&partner=JM&partner=JO&partner=JP&partner=KE&partner=KG&partner=KH&partner=KP&partner=KR&partner=KW&partner=KZ&partner=LA&partner=LB&partner=LI&partner=LK&partner=LR&partner=LT&partner=LU&partner=LV&partner=LY&partner=MA&partner=MD&partner=ME&partner=MG&partner=MH&partner=MK&partner=MM&partner=MN&partner=MR&partner=MT&partner=MU&partner=MX&partner=MY&partner=MZ&partner=NA&partner=NC&partner=NE&partner=NG&partner=NL&partner=NO&partner=NP&partner=NSP&partner=NZ&partner=OM&partner=PA&partner=PE&partner=PG&partner=PH&partner=PK&partner=PL&partner=PT&partner=QA&partner=RO&partner=RS&partner=RU&partner=SA&partner=SD&partner=SE&partner=SG&partner=SI&partner=SK&partner=SL&partner=SN&partner=SS&partner=ST&partner=SY&partner=TG&partner=TH&partner=TJ&partner=TL&partner=TM&partner=TN&partner=TOTAL&partner=TR&partner=TT&partner=TW&partner=TZ&partner=UA&partner=UG&partner=UK&partner=US&partner=UY&partner=UZ&partner=VE&partner=VG&partner=VN&partner=XK&partner=YE&partner=ZA&lang=EN"

# API abrufen
response = requests.get(url)
response.raise_for_status()

# JSON-stat als String für pyjstat
json_string = json.dumps(response.json())

# JSON-stat to DataFrame
dataset = pyjstat.Dataset.read(json_string)
kerosene_jetfuel_imports_df = dataset.write('dataframe')

# Defining values to be numeric to avoid issues later on
kerosene_jetfuel_imports_df['value'] = pd.to_numeric(kerosene_jetfuel_imports_df['value'], errors='coerce')

# Data Cleaning and Transformation
kerosene_jetfuel_imports_df = kerosene_jetfuel_imports_df.dropna(subset=["value"])
kerosene_jetfuel_imports_df = kerosene_jetfuel_imports_df.rename(columns={"value": "Kerosene Jet Fuel Imports (in TSD Tonnes)", "Time": "Year"})
kerosene_jetfuel_imports_df = kerosene_jetfuel_imports_df.drop(columns=["Time frequency", "Unit of measure"])
kerosene_jetfuel_imports_df["Country Code"] = kerosene_jetfuel_imports_df["Geopolitical entity (reporting)"].map(country_map)
kerosene_jetfuel_imports_df["YearCountryKey"] = kerosene_jetfuel_imports_df["Year"].astype(str) + kerosene_jetfuel_imports_df["Country Code"]

print(kerosene_jetfuel_imports_df.head())

  Standard international energy product classification (SIEC)  \
0                             Kerosene-type jet fuel            
1                             Kerosene-type jet fuel            
2                             Kerosene-type jet fuel            
3                             Kerosene-type jet fuel            
4                             Kerosene-type jet fuel            

  Geopolitical entity (partner)            Geopolitical entity (reporting)  \
0                       Belgium  European Union - 27 countries (from 2020)   
1                       Belgium  European Union - 27 countries (from 2020)   
2                       Belgium  European Union - 27 countries (from 2020)   
3                       Belgium  European Union - 27 countries (from 2020)   
4                       Belgium  European Union - 27 countries (from 2020)   

   Year  Kerosene Jet Fuel Imports (in TSD Tonnes) Country Code YearCountryKey  
0  1990                                      442.0    EU27-

In [40]:
kerosene_jetfuel_imports_df.to_csv("Kerosene_Jet_Fuel_Imports.csv", index=False, sep=';', decimal=',')

## Bisher erstellte Data Frames übersicht

##### Money Market Interest Rate Yearly (Yearly, Total): mm_ir_yy_df
##### GDP per capita data (Yearly, Total): gdp_pc_df
##### Imports of Goods and Services as % to GDP (Yearly, Categorical, total included to economic df): imports_percgdp_df
##### Foreign Value Added (Yearly, Categorical, Value devided by importing nation, total included to economic df): fva_df
##### Harmonized Index Consumer Prices (annual average rate of change) (Yearly, Total): hicp_df
##### Investments as Percentage to GDP (Yearly, Categorical, Included in Eco): investments_df
##### Government Spending per National Account Category (Yearly, Categorical, Included in Eco): gov_de
##### Hours Worked (Yearly, Categorical, Included in Eco): labor_df

##### HICP contribution to Inflation Rate (in %) EUR Area Monthly Data (Yearly, Categorical): hicp_perc_infl_eur_df
##### Consumption (in percentages) (Yearly Categorical, cannot be included in Eco): consumption_df
##### Gross domestic product (GDP) and main components (output, expenditure and income) (Yearly, Categorical, Percentage, cannot be included in Eco): gdp_components_df
##### Money Supply in ENP COuntries: money_supply_enp_df
##### Money Supply in Euro Area: money_supply_eur_df
